## Verify the JDBC driver

```
Verify that PostgreSQL jar is there. Anything inside /opt/spark3/jars will automatically come in classpath of Spark3.

cd /opt/spark3/jars
ls -lh postgresql-42.7.2.jar
```



## Copy the dataset to `/tmp`

```
cd ~/Test-Sandbox/Spark-Postgres/

# Copy data to /tmp
cp salaries.txt /tmp
```


## Connect to PostgreSQL Database container

```
# Perform following activities on PostgreSQL Prompt
docker exec \
    -it cluster_util_db \
    psql -U postgres \
    -W
    
# Password: itversity
```


## List databases

```
# Inside your interactive psql session in the container, execute:

\l
```

## Create a database `test` and then a table `salaries`

```
# Inside your interactive psql session in the container, execute:

DROP DATABASE IF EXISTS test;
\l
CREATE DATABASE test;

# Change to database
\c test;
# Password = itversity
\l

# List tables
\dt

DROP TABLE IF EXISTS salaries;

CREATE TABLE salaries (
    gender VARCHAR(1),
    age INT,
    salary DOUBLE PRECISION,
    zipcode INT
);
```

## Describe newly created table
```
# Inside your interactive psql session in the container, execute:

\d salaries;

```

## Copy File to Container
Copy salaries.txt from your host machine into the /tmp/ directory of the cluster_util_db container

```
# Give following command on the bash shell

docker cp /tmp/salaries.txt cluster_util_db:/tmp/salaries.txt
```

## Verify the file on docker container
To list the file inside the cluster_util_db container, use the docker exec command to run ls directly:

```
# Give following command on the bash shell

docker exec -it cluster_util_db ls -l /tmp/salaries.txt
```

## Run \copy Command

```
# Inside your interactive psql session in the container, execute:

\copy salaries(gender, age, salary, zipcode) FROM '/tmp/salaries.txt' WITH (FORMAT csv, DELIMITER ',');
```

## Add Primary Key to the table
Copy data to table and alter table to add a primary key

```
# Inside your interactive psql session in the container, execute:

ALTER TABLE salaries ADD COLUMN id SERIAL PRIMARY KEY;

```

## Verify the Primary key column and verify data

```
# Inside your interactive psql session in the container, execute:

\d salaries;
  
SELECT * FROM salaries limit 10;

```

## Add jdbc driver on the cluster

```
# Give following command on the bash shell

hdfs dfs -put /opt/spark3/jars/postgresql-42.7.2.jar /spark3-jars/
hdfs dfs -ls /spark3-jars/postgresql-42.7.2.jar
```

In [1]:
# Explicitly Point Spark to the Correct Python Executable
# If you already have numpy installed in a specific environment (like conda or a virtualenv), but PySpark is defaulting to /usr/bin/python3 or system Python, 
# explicitly set the Python environment variables before initializing SparkContext:

import os
import sys

# Set these before initializing SparkContext (sc)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
from pyspark.sql import SparkSession

In [3]:
# If you are developing an application you should use local mode. 
# Running in yarn mode is bit time consuming than running in local mode
# If you still insist to run in yarn mode then make sure that
# 1. You assigned more than one processor in the VM settings
# 2. Run only one kernel at a time

In [4]:
spark = SparkSession. \
    builder. \
    enableHiveSupport(). \
    appName('Demo'). \
    master('yarn'). \
    getOrCreate()

In [5]:
# spark = SparkSession. \
#     builder. \
#     enableHiveSupport(). \
#     appName('Demo'). \
#     master('local[*]'). \
#     getOrCreate()

In [6]:
sc = spark.sparkContext

## Pyspark working with PostgreSQL


In [7]:
url = "jdbc:postgresql://localhost:6432/test"
# url = "jdbc:mysql://127.0.0.1:3306/test?useSSL=false&allowPublicKeyRetrieval=true"
driver = "org.postgresql.Driver"
# driver = "com.mysql.jdbc.Driver"
user = "postgres"
#user = "bigdata"
password = "itversity"
#password = "Bigdata@123"

# https://youtu.be/ray3YvnIohM

In [8]:
df =  spark.read\
    .format("jdbc")\
    .option("driver", driver)\
    .option("url", url)\
    .option("user", user)\
    .option("password", password)\
    .option("dbtable", "salaries")\
    .load()
df.count()

50

In [9]:
df =  spark.read\
    .format("jdbc")\
    .option("driver", driver)\
    .option("url", url)\
    .option("user", user)\
    .option("password", password)\
    .option("dbtable", "salaries")\
    .load()
df.show(5)

+------+---+-------+-------+---+
|gender|age| salary|zipcode| id|
+------+---+-------+-------+---+
|     F| 66|41000.0|  95103|  1|
|     M| 40|76000.0|  95102|  2|
|     F| 58|95000.0|  95103|  3|
|     F| 68|60000.0|  95105|  4|
|     M| 85|14000.0|  95102|  5|
+------+---+-------+-------+---+
only showing top 5 rows



## Reading From Database in Parallel

When we are reading large table, we would like to read that in parallel. This will dramatically improve read performance. We can pass “numPartitions” option to spark read function which will decide parallelism in reading data.

In [10]:
df =  spark.read\
    .format("jdbc")\
    .option("driver", driver)\
    .option("url", url)\
    .option("user", user)\
    .option("password", password)\
    .option("dbtable", "salaries")\
    .option("numPartitions", 10)\
    .load()
 
df.rdd.getNumPartitions()

1

In our case, it will still show as 1 partition only. This is because we do not have enough data to create 10 different partitions

## Quit the psql prompt
Quit the psql prompt

```
# Inside your interactive psql session in the container, execute:

\q

```

## Perform cleaning activities

```
# Give following command on the bash shell

hdfs dfs -rm /spark3-jars/postgresql-42.7.2.jar
rm /tmp/salaries.txt
docker exec -it cluster_util_db rm /tmp/salaries.txt
docker exec -it cluster_util_db ls /tmp/salaries.txt

```